<a href="https://colab.research.google.com/github/Amruth-U-tech/DL-Journey/blob/main/05-LSTMs/BidirectionalLSTM-1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install yfinance

In [3]:
import yfinance as yf
import numpy as np
import pandas as pd #for data handling or to read
import matplotlib.pyplot as plt #to print any 2D instance
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM,Add, Bidirectional, Input  #we got two RNNs here
from tensorflow.keras.optimizers import Adam

In [4]:
df = yf.download("AAPL",period="5y",auto_adjust=True)[["Close"]]  #we are dowloding "AAPL" which mean aplle stock market data of period 5years

#with the target variable as close which means there is a feature in that data known as close which shos the closing price of the stock and that Y
df.head(10)

[*********************100%***********************]  1 of 1 completed


Price,Close
Ticker,AAPL
Date,
2021-03-17,121.515694
2021-03-18,117.395691
2021-03-19,116.869736
2021-03-22,120.181320
2021-03-23,119.353401
2021-03-24,116.967110
2021-03-25,117.454124
2021-03-26,118.058006


In [5]:
scaler = MinMaxScaler()
scaled_data = scaler.fit_transform(df)
scaled_data

array([[2.79863808e-02],
       [3.62790093e-03],
       [5.18323112e-04],
       ...,
       [7.88326963e-01],
       [8.04290106e-01],
       [8.07778852e-01]])

In [6]:
from pandas.core.window.rolling import Window
#now because this is Sequential data we make our x and Y in such a way itself
#such that some input X1 will give some input Y1 then that Y1 is included with X1 by removing previous data of X1 hence X2=X1-X1[0]+Y1 we get Y2

def create_sequence(data,window=30):
  X,Y = [],[]
  for i in range(window,len(data)):
    X.append(data[i-window:i])
    Y.append(data[i])
  return np.array(X),np.array(Y)

Window_size = 30
x,y = create_sequence(scaled_data,window=Window_size)

In [7]:
x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.2,shuffle=False)
print(x_train.shape)
print(x_test.shape)

(980, 30, 1)
(246, 30, 1)


In [8]:
def complie_and_train(model,name,epochs=10,batch_size=32):
    model.compile(optimizer='adam',loss='mse')
    history = model.fit(x_train,y_train,epochs=epochs,batch_size=batch_size,verbose=1)

    print(f"{name} trained")
    return history

In [9]:
Window_size=30
batch_size = 32
epochs =10
model_stackedLSTM = Sequential([Input(shape=(Window_size,1)),LSTM(50,return_sequences=True),LSTM(50),Dense(1)])
model_stackedLSTM.summary()

hist_stackedLSTM = complie_and_train(model_stackedLSTM,"stackedLSTM",epochs=epochs,batch_size=batch_size)

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 30, 50)         │        10,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 50)             │        20,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │            51 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 30,651 (119.73 KB)

 Trainable params: 30,651 (119.73 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
31/31 ━━━━━━━━━━━━━━━━━━━━ 6s 8ms/step - loss: 0.0290
Epoch 2/10
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.0024
Epoch 3/10
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0016
Epoch 4/10
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0015
Epoch 5/10
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.0014
Epoch 6/10
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0014
Epoch 7/10
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0013
Epoch 8/10
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0013
Epoch 9/10
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0012
Epoch 10/10
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0011
stackedLSTM trained


In [10]:
#model prediction
prediction = model_stackedLSTM.predict(x_test)
prediction=scaler.inverse_transform(prediction)
y_test1 = scaler.inverse_transform(y_test)

for i in range(len(prediction)):
  print(f"prdeiction: {prediction[i][0]:.2f}, actual:{y_test1[i][0]:.2f}")

8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step
prdeiction: 207.08, actual:222.78
prdeiction: 207.69, actual:220.57
prdeiction: 208.62, actual:222.88
prdeiction: 209.89, actual:216.95
prdeiction: 210.87, actual:221.17
prdeiction: 211.91, actual:222.22
prdeiction: 212.99, actual:222.92
prdeiction: 214.06, actual:202.31
prdeiction: 213.54, actual:187.56
prdeiction: 211.01, actual:180.67
prdeiction: 206.85, actual:171.67
prdeiction: 201.26, actual:197.99
prdeiction: 196.95, actual:189.59
prdeiction: 193.13, actual:197.29
prdeiction: 190.49, actual:201.64
prdeiction: 189.12, actual:201.26
prdeiction: 188.69, actual:193.43
prdeiction: 188.43, actual:196.13
prdeiction: 188.50, actual:192.32
prdeiction: 188.53, actual:198.87
prdeiction: 188.98, actual:203.71
prdeiction: 190.01, actual:207.47
prdeiction: 191.63, actual:208.37
prdeiction: 193.61, actual:209.23
prdeiction: 195.77, actual:210.29
prdeiction: 198.00, actual:211.58
prdeiction: 200.20, actual:212.39
prdeiction: 202.29, actual:204.46
prdeiction

**bidirectional LSTM**

In [11]:
model_bid = Sequential([Input(shape=(Window_size,1)),Bidirectional(LSTM(50)),Dense(1)])
model_bid.summary()

hist_bid = complie_and_train(model_bid,"bidirectional LSTM",epochs=epochs,batch_size=batch_size)

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ bidirectional (Bidirectional)   │ (None, 100)            │        20,800 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           101 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 20,901 (81.64 KB)

 Trainable params: 20,901 (81.64 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
31/31 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - loss: 0.0153
Epoch 2/10
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0028
Epoch 3/10
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0022
Epoch 4/10
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0020
Epoch 5/10
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.0018
Epoch 6/10
31/31 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - loss: 0.0016
Epoch 7/10
31/31 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - loss: 0.0013
Epoch 8/10
31/31 ━━━━━━━━━━━━━━━━━━━━ 1s 34ms/step - loss: 0.0011
Epoch 9/10
31/31 ━━━━━━━━━━━━━━━━━━━━ 1s 30ms/step - loss: 0.0011
Epoch 10/10
31/31 ━━━━━━━━━━━━━━━━━━━━ 1s 23ms/step - loss: 9.7315e-04
bidirectional LSTM trained


In [12]:
prediction = model_bid.predict(x_test)
prediction=scaler.inverse_transform(prediction)
y_test1 = scaler.inverse_transform(y_test)

for i in range(len(prediction)):
  print(f"prdeiction: {prediction[i][0]:.2f}, actual:{y_test1[i][0]:.2f}")

8/8 ━━━━━━━━━━━━━━━━━━━━ 1s 57ms/step
prdeiction: 213.59, actual:222.78
prdeiction: 215.78, actual:220.57
prdeiction: 217.92, actual:222.88
prdeiction: 220.32, actual:216.95
prdeiction: 221.83, actual:221.17
prdeiction: 223.45, actual:222.22
prdeiction: 225.04, actual:222.92
prdeiction: 226.49, actual:202.31
prdeiction: 225.05, actual:187.56
prdeiction: 220.97, actual:180.67
prdeiction: 215.12, actual:171.67
prdeiction: 207.66, actual:197.99
prdeiction: 203.56, actual:189.59
prdeiction: 199.56, actual:197.29
prdeiction: 196.90, actual:201.64
prdeiction: 195.79, actual:201.26
prdeiction: 195.57, actual:193.43
prdeiction: 195.03, actual:196.13
prdeiction: 195.08, actual:192.32
prdeiction: 194.60, actual:198.87
prdeiction: 195.39, actual:203.71
prdeiction: 197.19, actual:207.47
prdeiction: 199.72, actual:208.37
prdeiction: 202.83, actual:209.23
prdeiction: 205.81, actual:210.29
prdeiction: 208.67, actual:211.58
prdeiction: 211.46, actual:212.39
prdeiction: 213.87, actual:204.46
prdeiction

**multilayer LSTM **

**stateful LSTM**